In [ ]:
#!/usr/bin/env python3
"""
Multipart uploader for very large files.
Notebook-friendly version using argparse(parse_args(args=[])).
"""

import argparse
import logging
import math
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import boto3
from botocore.config import Config
from botocore.exceptions import (
    BotoCoreError,
    ClientError,
    ReadTimeoutError,
    ConnectTimeoutError,
)

def parse_args(args=None):
    parser = argparse.ArgumentParser(description="Multipart uploader")
    parser.add_argument("-b", "--bucket", required=True)
    parser.add_argument("-f", "--file", dest="file_path", required=True)
    parser.add_argument("-k", "--key", required=True)
    parser.add_argument("-c", "--chunk-size", type=int, default=50 * 1024 * 1024)
    parser.add_argument("-a", "--access_key", default=os.environ.get("AWS_ACCESS_KEY_ID"))
    parser.add_argument("-s", "--secret_key", default=os.environ.get("AWS_SECRET_ACCESS_KEY"))
    parser.add_argument("-e", "--endpoint", default=os.environ.get("S3_ENDPOINT"))
    parser.add_argument("-r", "--region", default=os.environ.get("S3_REGION", "us-east-1"))
    parser.add_argument("-m", "--max-retries", type=int, default=5)
    return parser.parse_args(args=args)

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

class LargeMultipartUploader:
    def __init__(self, *, file_path, bucket, key, region, access_key, secret_key, endpoint, part_size=50*1024*1024, max_retries=5):
        self.file_path = file_path
        self.bucket = bucket
        self.key = key
        self.region = region
        self.access_key = access_key
        self.secret_key = secret_key
        self.endpoint = endpoint
        self.part_size = part_size
        self.max_retries = max_retries
        self.progress_lock = Lock()
        self.parts_completed = 0
        self.session = boto3.session.Session(
            aws_access_key_id=self.access_key,
            aws_secret_access_key=self.secret_key,
            region_name=self.region,
        )
        self.botocore_cfg = Config(region_name=self.region, retries={"max_attempts": self.max_retries, "mode": "standard"})
        self.s3 = self.session.client("s3", config=self.botocore_cfg, endpoint_url=self.endpoint)
        self.upload_id = None

    def call_with_retry(self, desc, func):
        for attempt in range(1, self.max_retries+1):
            try:
                return func()
            except Exception as exc:
                if attempt == self.max_retries:
                    raise
                time.sleep(2**attempt)

    def upload_part(self, *, part_number, offset, bytes_to_read):
        with open(self.file_path, "rb") as f:
            f.seek(offset)
            data = f.read(bytes_to_read)
        resp = self.s3.upload_part(Bucket=self.bucket, Key=self.key, PartNumber=part_number, UploadId=self.upload_id, Body=data)
        return {"PartNumber": part_number, "ETag": resp["ETag"]}

    def upload(self):
        file_size = os.path.getsize(self.file_path)
        total_parts = math.ceil(file_size / self.part_size)
        self.upload_id = self.call_with_retry("create_multipart_upload", lambda: self.s3.create_multipart_upload(Bucket=self.bucket, Key=self.key))["UploadId"]
        parts = []
        with ThreadPoolExecutor(max_workers=4) as exe:
            futures = []
            for pn in range(1, total_parts+1):
                offset = (pn-1) * self.part_size
                size = min(self.part_size, file_size-offset)
                futures.append(exe.submit(self.upload_part, part_number=pn, offset=offset, bytes_to_read=size))
            for f in as_completed(futures): parts.append(f.result())

        parts_sorted = sorted(parts, key=lambda x: x["PartNumber"])
        self.call_with_retry("complete_multipart_upload", lambda: self.s3.complete_multipart_upload(
            Bucket=self.bucket,
            Key=self.key,
            UploadId=self.upload_id,
            MultipartUpload={"Parts": parts_sorted},
        ))
        print("Upload complete!")

# Notebook config section
BUCKET = ""
FILE_PATH = ""
KEY = ""
REGION = os.environ.get("S3_REGION", "us-east-1")
ENDPOINT = os.environ.get("S3_ENDPOINT", "")
ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
CHUNK_SIZE = 50 * 1024 * 1024

if all([BUCKET, FILE_PATH, KEY, ENDPOINT, ACCESS_KEY, SECRET_KEY]):
    uploader = LargeMultipartUploader(
        file_path=FILE_PATH,
        bucket=BUCKET,
        key=KEY,
        region=REGION,
        access_key=ACCESS_KEY,
        secret_key=SECRET_KEY,
        endpoint=ENDPOINT,
        part_size=CHUNK_SIZE,
    )
    # uploader.upload()   # Uncomment to run
else:
    print("⚠️ Fill in BUCKET, FILE_PATH, KEY, ENDPOINT, ACCESS_KEY, SECRET_KEY.")
